<a href="https://colab.research.google.com/github/hamshini1413/hamshini_gen_ai_foundation/blob/main/15_Enterprise_LLM_Gateway_with_Dynamic_Rate_Limiting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install fastapi uvicorn nest_asyncio pyngrok requests prometheus_client

In [2]:
import random
import time
import nest_asyncio

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from prometheus_client import Counter, Histogram, generate_latest

from pyngrok import ngrok

import uvicorn

In [3]:
app = FastAPI(title="Enterprise LLM Gateway")

In [4]:
REQUEST_COUNT = Counter(
    "gateway_requests_total",
    "Total Requests"
)

LATENCY = Histogram(
    "gateway_latency_seconds",
    "Gateway Latency"
)

In [5]:
class Prompt(BaseModel):
    text: str

In [6]:
TOKENS = 10
MAX_TOKENS = 10
LAST = time.time()

def allow_request():

    global TOKENS
    global LAST

    now = time.time()

    refill = (now - LAST) * 2

    TOKENS = min(MAX_TOKENS, TOKENS + refill)

    LAST = now

    if TOKENS >= 1:
        TOKENS -= 1
        return True

    return False

In [7]:
def primary_model(prompt):

    if random.random() < 0.30:
        raise Exception("Primary model failed")

    return "Primary Model Response: " + prompt

In [8]:
def backup_model(prompt):

    return "Fallback Model Response: " + prompt

In [9]:
@app.post("/chat")
def chat(data: Prompt):

    REQUEST_COUNT.inc()

    if not allow_request():

        raise HTTPException(
            status_code=429,
            detail="Rate Limit Exceeded"
        )

    start = time.time()

    try:

        result = primary_model(data.text)

        provider = "Primary"

    except:

        result = backup_model(data.text)

        provider = "Fallback"

    latency = time.time() - start

    LATENCY.observe(latency)

    return {
        "provider": provider,
        "latency": latency,
        "response": result
    }

In [10]:
@app.get("/metrics")
def metrics():

    return generate_latest()

In [11]:
import nest_asyncio
import asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [531]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [531]


In [14]:
result = chat(Prompt(text="Explain AI"))

print(result)

{'provider': 'Primary', 'latency': 6.9141387939453125e-06, 'response': 'Primary Model Response: Explain AI'}
